# 01 Data Merging

Notebook นี้รวมข้อมูล raw หลายตารางให้กลายเป็นไฟล์เดียวต่อ 1 match สำหรับใช้ใน EDA และ modeling


## 1. Import Library

โหลด `pandas` สำหรับอ่าน CSV, merge ตาราง, pivot ข้อมูล และบันทึกผลลัพธ์


In [1]:
import pandas as pd

## 2. Load Player-Level Tables

อ่านตารางสถิติผู้เล่นจาก `MatchStatsTbl.csv` และตาราง mapping ผู้เล่นกับ match/champion จาก `SummonerMatchTbl.csv`


In [2]:
stats_df = pd.read_csv('data/t1_raw/match_details/MatchStatsTbl.csv')
summoner_match_df = pd.read_csv('data/t1_raw/match_details/SummonerMatchTbl.csv')

## 3. Attach Match And Champion IDs

merge สถิติผู้เล่นกับข้อมูล `MatchFk` และ `ChampionFk` โดยใช้ `SummonerMatchFk` เทียบกับ `SummonerMatchId`


In [3]:
merged_df = pd.merge(
    stats_df, 
    summoner_match_df[['SummonerMatchId', 'MatchFk', 'ChampionFk']], 
    left_on='SummonerMatchFk', 
    right_on='SummonerMatchId', 
    how='left'
)

## 4. Convert Player Rows Into One Match Row

เปลี่ยนโครงสร้างข้อมูลจากเดิมที่แยกรายผู้เล่น (Player-Level) ให้รวมอยู่ในแถวเดียวต่อหนึ่งแมตช์ (Match-Level) โดยมีขั้นตอนดังนี้:
- Indexing : จัดเรียงรายชื่อผู้เล่นในแต่ละแมตช์และกำหนดรหัส P1 ถึง P10 เพื่อใช้เป็นตัวแทนตำแหน่งผู้เล่น
- Pivoting : ทำการ Pivot ข้อมูลเพื่อให้ 1 แมตช์แสดงผลเพียง 1 แถว
- Rename Columns: เพิ่มคำต่อท้าย ในแต่ละคอลัมน์ตามรหัสผู้เล่น เช่น _P1, _P2 เพื่อแยกแยะข้อมูลรายบุคคลในแถวนั้น


In [4]:
merged_df = merged_df.sort_values(by=['MatchFk', 'SummonerMatchId'])

merged_df['Player_No'] = merged_df.groupby('MatchFk').cumcount() + 1
merged_df['Player_No'] = 'P' + merged_df['Player_No'].astype(str)

cols_to_pivot = ['ChampionFk', 'Lane', 'TotalGold', 'MinionsKilled', 'kills', 'deaths', 'assists', 'DmgDealt', 'DmgTaken', 'TurretDmgDealt', 'PrimaryKeyStone', 'SummonerSpell1', 'SummonerSpell2', 'Win']
df_1_row = merged_df.pivot(index='MatchFk', columns='Player_No', values=cols_to_pivot)

df_1_row.columns = [f"{col[0]}_{col[1]}" for col in df_1_row.columns]
df_1_row = df_1_row.reset_index()

df_1_row['TeamBlue_Win'] = df_1_row['Win_P1']

## 5. Remove Player-Level Win Columns

ทำการลบ Columns ผลการชนะรายบุคคล (Win_P1 ถึง Win_P10) ที่เกิดขึ้นหลังจากการ Pivot ข้อมูล เนื่องจากเป็นข้อมูลชุดเดียวกับผลการแข่งขันในระดับทีม (Team-Level Win) เพื่อลดความซ้ำซ้อนของฟีเจอร์และป้องกันปัญหาในการฝึกสอนโมเดลในภาหลัง


In [5]:
drop_cols = [col for col in df_1_row.columns if col.startswith('Win_P')]
df_1_row = df_1_row.drop(columns=drop_cols)

## 6. Add Team-Level Objective And Result Data

อ่านไฟล์ `TeamMatchTbl.csv` และทำการ merge เข้ากับข้อมูลราย Match (df_1_row) เพื่อเพิ่ม Objective Counts (Baron, Rift, Dragon, Tower, Kills) และ Target `BlueWin` / `RedWin`


In [6]:
team_df = pd.read_csv('data/t1_raw/match_details/TeamMatchTbl.csv')

final_df = pd.merge(
    df_1_row, 
    team_df, 
    on='MatchFk', 
    how='inner'   
)

if 'TeamBlue_Win' in final_df.columns:
    final_df = final_df.drop(columns=['TeamBlue_Win'])

## 7. Add Match Metadata

อ่านไฟล์ `MatchTbl.csv` แล้ว merge Metadata เช่น `Patch`, `QueueType`, `RankFk`, และ `GameDuration` เข้ากับ dataset (final_df)


In [7]:
mdf = pd.read_csv("data/t1_raw/match_details/MatchTbl.csv")

merged = final_df.merge(mdf, left_on='MatchFk', right_on='MatchId', how='left')

if 'MatchId' in merged.columns:
    merged = merged.drop(columns='MatchId')

merged.head()

,MatchFk,ChampionFk_P1,ChampionFk_P10,ChampionFk_P2,ChampionFk_P3,ChampionFk_P4,ChampionFk_P5,ChampionFk_P6,ChampionFk_P7,ChampionFk_P8,...,RedRiftHeraldKills,RedDragonKills,RedTowerKills,RedKills,RedWin,BlueWin,Patch,QueueType,RankFk,GameDuration
0,SG2_136048780,266,117,32,90,202,63,164,254,3,...,0,1,0,3,0,1,16.4.746.5697,CLASSIC,1,1488
1,SG2_136055679,30,800,420,523,8,38,13,61,223,...,0,1,1,23,1,0,16.4.746.5697,URF,1,714
2,SG2_136057602,38,21,10,235,517,157,101,126,107,...,3,1,5,28,0,1,16.4.746.5697,URF,1,1418
3,SG2_136061350,222,101,3,141,106,518,13,254,25,...,1,0,1,18,1,0,16.4.746.5697,URF,1,1153
4,SG2_136061888,92,147,82,238,29,45,122,32,86,...,0,1,0,6,0,1,16.4.746.5697,CLASSIC,1,2542


## 8. Create Working Copy

คัดลอก merged dataframe เป็น `df` เพื่อใช้ตรวจและ clean ข้อมูลก่อนบันทึก


In [8]:
df = merged.copy()

## 9. Check Invalid Target Rows

ตรวจแถวที่ `BlueWin` และ `RedWin` มีค่าเท่ากัน ซึ่งเป็นกรณี Target ไม่สอดคล้องกันสำหรับ binary classification


In [9]:
df[df["BlueWin"] == df["RedWin"]]

,MatchFk,ChampionFk_P1,ChampionFk_P10,ChampionFk_P2,ChampionFk_P3,ChampionFk_P4,ChampionFk_P5,ChampionFk_P6,ChampionFk_P7,ChampionFk_P8,...,RedRiftHeraldKills,RedDragonKills,RedTowerKills,RedKills,RedWin,BlueWin,Patch,QueueType,RankFk,GameDuration
2503,SG2_138840040,17,NaN,120,80,29,78,19,30,69,...,0,0,2,11,0,0,16.5.749.6037,SWIFTPLAY,1,1083
17392,SG2_140240109,157,497,121,800,22,35,39,77,127,...,0,1,0,7,0,0,16.5.751.9084,CLASSIC,1,830
54034,SG2_142512774,910,36,202,950,30,58,350,89,4,...,0,0,0,42,0,0,16.6.756.931,CHERRY,1,1568
56824,SG2_142567875,875,53,102,157,145,518,150,234,268,...,0,0,0,3,0,0,16.6.756.931,CLASSIC,1,1707
59648,SG2_142636380,14,29,11,166,81,89,102,517,131,...,2,1,0,17,0,0,16.6.756.931,CLASSIC,1,1099
65915,SG2_142846783,950,37,59,45,202,111,106,56,13,...,3,1,0,7,0,0,16.6.756.931,CLASSIC,1,1148
66402,SG2_142864733,68,235,203,61,804,25,41,233,166,...,0,1,0,6,0,0,16.6.756.931,CLASSIC,1,1576
73056,SG2_143068018,777,412,64,101,51,143,24,102,950,...,0,1,0,5,0,0,16.6.756.9613,CLASSIC,1,1808
73471,SG2_143092692,4,117,91,711,21,111,82,107,238,...,2,0,0,4,0,0,16.6.756.9613,CLASSIC,1,2019
75383,SG2_143197663,350,114,517,91,429,360,80,427,59,...,0,0,0,34,0,0,16.6.756.9613,CHERRY,1,1677


## 10. Check Missing Champion IDs

ตรวจ missing values ใน `ChampionFk_P1` ถึง `ChampionFk_P10` เพราะแต่ละ match ควรมี champion ครบ 10 คนห้ามขาด


In [10]:
champ_fk_cols = [f"ChampionFk_P{p}" for p in range(1, 11)]
null_check = df[champ_fk_cols].isnull().sum()
null_check

ChampionFk_P1     0
ChampionFk_P2     2
ChampionFk_P3     2
ChampionFk_P4     3
ChampionFk_P5     4
ChampionFk_P6     4
ChampionFk_P7     5
ChampionFk_P8     5
ChampionFk_P9     7
ChampionFk_P10    7
dtype: int64

## 11. Drop Rows With Missing Champion IDs

ลบ Match ที่ Champion ID ไม่ครบออก เพื่อให้ข้อมูลผู้เล่นทั้งสองทีมสมบูรณ์ก่อนนำไปใช้ต่อ




In [11]:
df = df.dropna(subset=champ_fk_cols).reset_index(drop=True)

## 12. Drop Unused Columns

ทำการลบ column ที่ซ้ำกับข้อมูลผู้เล่นที่ pivot แล้ว หรือไม่ได้ใช้ใน transformed dataset เช่น ban/pick duplicate columns, `GameDuration`, `RankFk`, และ `TeamID`


In [12]:
drop_cols = [
    "B1Champ", "B2Champ", "B3Champ", "B4Champ", "B5Champ",
    "R1Champ", "R2Champ", "R3Champ", "R4Champ", "R5Champ",
    "GameDuration", "RankFk", "TeamID",
]

df = df.drop(columns=drop_cols, errors="ignore")

## 13. Preview Final Data

แสดง Dataframe หลัง merge และ clean เพื่อตรวจ column และจำนวนข้อมูลก่อน save


In [13]:
df

,MatchFk,ChampionFk_P1,ChampionFk_P10,ChampionFk_P2,ChampionFk_P3,ChampionFk_P4,ChampionFk_P5,ChampionFk_P6,ChampionFk_P7,ChampionFk_P8,...,BlueKills,RedBaronKills,RedRiftHeraldKills,RedDragonKills,RedTowerKills,RedKills,RedWin,BlueWin,Patch,QueueType
0,SG2_136048780,266,117,32,90,202,63,164,254,3,...,3,0,0,1,0,3,0,1,16.4.746.5697,CLASSIC
1,SG2_136055679,30,800,420,523,8,38,13,61,223,...,9,0,0,1,1,23,1,0,16.4.746.5697,URF
2,SG2_136057602,38,21,10,235,517,157,101,126,107,...,16,0,3,1,5,28,0,1,16.4.746.5697,URF
3,SG2_136061350,222,101,3,141,106,518,13,254,25,...,22,0,1,0,1,18,1,0,16.4.746.5697,URF
4,SG2_136061888,92,147,82,238,29,45,122,32,86,...,6,0,0,1,0,6,0,1,16.4.746.5697,CLASSIC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91406,SG2_144017659,777,43,77,8,157,516,11,876,910,...,11,0,0,0,0,16,0,1,16.6.756.9613,SWIFTPLAY
91407,SG2_144017694,50,37,107,910,202,555,79,136,4,...,9,0,0,1,0,12,1,0,16.6.756.9613,SWIFTPLAY
91408,SG2_144018229,235,115,63,11,235,150,201,31,17,...,10,0,0,0,0,33,0,1,16.6.756.9613,CHERRY
91409,SG2_144018321,157,800,67,202,145,53,360,56,3,...,21,0,0,1,1,13,0,1,16.6.756.9613,SWIFTPLAY


## 14. Save Transformed Dataset

บันทึกไฟล์ `data/t2_transformed/merged_v1.csv` สำหรับใช้ใน Notebook EDA และ modeling ต่อไป


In [14]:
df.to_csv('data/t2_transformed/merged_v1.csv', index=False)